In [40]:
#First get the useful web urls that we can use from guardian.com

import requests
from bs4 import BeautifulSoup
import time
import logging
import json

logging.basicConfig(
        filename="get_webs.log",
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True)

def get_articles(soup) -> list:
    #Find all article links on that page that we can use
    links = []

    logging.info('request success, getting urls')

    for a in soup.find_all('a', href=True):
        if "/sport/2026/" in a['href']:
            links.append(a['href'])

    print(f'Found {len(links)} articles URLs')
    return links

def save_json(lists):
    data = {}
    for index, list in enumerate(lists):
        data[f'page{index + 1}'] = list

    logging.info('saving to json')
    
    with open('url_data.json','w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def main():
    #Add all the links in the list in future? for now, only guardian
    the_guardian_url = "https://www.theguardian.com/sport/winter-olympics"
    
    #I will get the first 5 pages, cuz I don't wanna scrap every single page.
    total_urls = 0
    each_page_lists = []
    for i in range(1,6):
        this_page_url = f"{the_guardian_url}?page={i}"
       
        try:
            logging.info(f'requesting {this_page_url}')

            response = requests.get(this_page_url)
            soup = BeautifulSoup(response.content, 'html.parser')
        
        except requests.exceptions.RequestException as e:
            print(f"error occured: {e}")
            logging.info(f'failed requesting')
            continue

        except Exception as e:
            print(f"error occured: {e}")
            logging.info(f'faield requesting')
            continue
        
        articles_list = get_articles(soup)
        each_page_lists.append(articles_list)
        total_urls += len(articles_list)

        #wait couple seconds before going next page (3sec)
        time.sleep(3)
    
    print(f'total articles: {total_urls}')
    logging.info(f"Total URLs collected: {total_urls}")
    print(f'will save the urls to json...')
    
    save_json(each_page_lists)
    print(f'saved')

main()

Found 19 articles URLs
Found 19 articles URLs
Found 21 articles URLs
Found 18 articles URLs
Found 17 articles URLs
total articles: 94
will save the urls to json...
saved


In [42]:
#dcr-130mj7b
#dcr-u0h1qy
import pprint
import requests
from bs4 import BeautifulSoup
import time
import logging
import csv
from datetime import datetime
import pandas as pd


the_guardian_url = "https://www.theguardian.com"

def web_scrapping(url, page_num, article_num):
    """
    Web scrapping. Will log every single time we access to the website. (Success or fail(error))

    url: url that we're trying to scrape
    page_num: which page we got that url
    article_num: nth article in that page

    """
    headers = {'User-Agent': 'MRU Data Science Student Project'}

    logging.basicConfig(
    filename='scraping_log.log', 
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True)

    time.sleep(4)
    
    try:
        logging.info(f"Starting request for: {url}")
        this_url = url
        response = requests.get(this_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        dict = {}
        dict['Article_ID'] = f'P{page_num}A{article_num}'
        dict['Source_ID'] = 'the guardian'
        dict['Title'] = soup.h1.text
        dict['Publish_Date'] = soup.find_all(class_='dcr-u0h1qy')[0].text
        dict['URL'] = this_url             
        dict['Article_Text'] = " ".join(p.get_text(strip=True) for p in soup.find_all("p", class_='dcr-130mj7b'))

        logging.info(f"Successfully scraped: {url}")

        return dict

    except requests.exceptions.HTTPError as e: 
        logging.error(f"HTTP Error for {url}: {e}")

    except requests.exceptions.ConnectionError as e:
        logging.error(f"Connection Error for {url}: {e}")

    except Exception as e:
        logging.error(f"Unexpected Error for {url}: {e}")
    
    return None

def analyze(data):
    #Keywords for the countries or entities under analysis
    target_keywords = {
    "Russia": ["Russia", "Russian", "Moscow", "AIN", "Neutral Athlete"],
    "Belarus": ["Belarus", "Belarusian", "Minsk"],
    "Israel": ["Israel", "Israeli", "Tel Aviv"]}

    # Political keywords for identifying framing
    political_keywords = ["war", "invasion", "sanction", "protest", "ban", "neutrality", "conflict"]

    url = data['URL']
    title = data['Title']
    content_to_check = (url + " " + title).lower()

    found_countries = ''
    
    for country, keywords in target_keywords.items():
        if any(kw.lower() in content_to_check for kw in keywords):
            found_countries += f'{country},'

    if found_countries == '': 
        return None
    
    text = data['Article_Text'].lower()

    mention_count = sum(text.count(kw.lower()) for kw in target_keywords)
    # has_political_context = any(p_kw.lower() in text for p_kw in political_keywords)
    has_political_context = sum(text.count(p_kw.lower()) for p_kw in political_keywords)

    if mention_count >= 2 and has_political_context:
        
        framing_analysis = {}
        framing_analysis['Article_ID'] = data['Article_ID']#primary/foreign key
        framing_analysis['Target_Nation'] = found_countries[:-1]
        framing_analysis['Frequency_of_Political_Terms'] = has_political_context

        return framing_analysis
    

    return None 


def update_acquisition_log(domain, records_count, errors_count, related_count):

    log_file = 'acquisition_log.csv'
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(log_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([timestamp, domain, records_count, errors_count, related_count])

def save_to_json(articles,analysis):
    
    with open("Articles.json", "w") as f:
        json.dump(articles, f, indent=4)

    with open("Source_ID.json", "w") as f:
        json.dump(analysis, f, indent=4)

def main():
    the_guardian_url = "https://www.theguardian.com"

    with open('url_data.json', 'r', encoding='utf-8') as f:
        page_urls = json.load(f)

    success_count = 0 #amount of success scrapping article
    errors_count = 0 #amount of error
    related_count = 0 #amount of url that relates to our question

    articles_ = []
    framing_analysis_ = []

    for page in page_urls: #page x
        for index, url in enumerate(page_urls[page], start=1): #num & url
           
            this_url = f"{the_guardian_url}{url}"
            result = web_scrapping(this_url, page[4:],index)

            if result is None:
                errors_count += 1
                continue

            success_count += 1
            analysis = analyze(result)

            if analysis is None: #Was not the article we we're looking for
                continue

            articles_.append(result)
            framing_analysis_.append(analysis)

        
    save_to_json(articles_, framing_analysis_)
    update_acquisition_log("theguardian.com", success_count, errors_count, related_count)
            
main()